#### Syntax 1: Pipeline API

- This is a high-level API.

Hugging Face automatically:

1. Downloads the model
2. Downloads the tokenizer
3. Loads the model
4. Creates a text-generation pipeline

In [4]:
from transformers import pipeline
pipe = pipeline("text-generation", model="openai-community/gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [14]:
output = pipe("What is Machine Learning", max_new_tokens=100, pad_token_id=50256)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [15]:
print(output[0]['generated_text'])

What is Machine Learning?

Machine learning is a new field that enables the creation of novel or better algorithms for data mining. Machine learning is a new field that enables the creation of novel or better algorithms for data mining. The most obvious use of machine learning is for data mining applications. As the popularity of this field increases, data mining applications may be able to grow quickly and efficiently by exploiting the potential of machine learning for both data mining and data storage. The most obvious use of machine learning is for data mining applications


#### Syntax 2: AutoTokenizer

- Only the tokenizer is loaded.

- No model is loaded.

- No text generation happens.

- You are loading only the component that converts text into tokens.

In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [2]:
tokenizer

GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [40]:
sentence = "unsure"
input_ids = tokenizer(sentence, return_tensors="pt")["input_ids"]
input_ids

tensor([[13271,   495]])

In [43]:
tokenizer.decode(input_ids[0][0]),tokenizer.decode(input_ids[0][1])

('uns', 'ure')

In [44]:
sentence = "unbelivable"
input_ids = tokenizer(sentence, return_tensors="pt").input_ids
input_ids

tensor([[  403,  6667, 21911]])

In [20]:
for token_id in input_ids[0]:
    print(tokenizer.decode(token_id))

un
bel
ivable


In [21]:
word = "homoscedasticity"
my_ids = tokenizer(word, return_tensors="pt").input_ids
my_ids

tensor([[26452,   418,   771,  3477,   414]])

In [45]:
tokenizer.decode(my_ids.squeeze())

'floccinaucinihilipilification'

In [46]:
word = "pneumonoultramicroscopicsilicovolcanoconiosis"
my_ids = tokenizer(word, return_tensors="pt").input_ids
# len(my_ids[0])
my_ids

tensor([[   79, 25668,   261, 25955,   859,  2500,  1416,   404,   873, 41896,
           709,   349,  5171, 36221, 42960]])

In [22]:
for token_id in my_ids.squeeze():
    print(tokenizer.decode(token_id))

hom
os
ced
astic
ity


In [23]:
sentence = "antidisestablishmentarianism"
token_ids = tokenizer(sentence, return_tensors="pt").input_ids
token_ids, len(token_ids[0])

(tensor([[  415, 29207, 44390,  3699,  1042]]), 5)

In [24]:
word = "floccinaucinihilipilification"

my_ids = tokenizer(word, return_tensors="pt").input_ids
my_ids, len(my_ids[0])

(tensor([[ 2704, 13966,   259, 14272,   259, 20898,   541,   346,  2649]]), 9)

In [25]:
for token_id in my_ids[0]:
    print(tokenizer.decode(token_id))

fl
occ
in
auc
in
ihil
ip
il
ification


#### Syntax3: AutoModelForCausalLM

| Component              | Purpose                               |
| ---------------------- | ------------------------------------- |
| `pipeline()`           | Loads tokenizer + model automatically |
| `AutoTokenizer`        | Converts text to tokens               |
| `AutoModelForCausalLM` | Loads LLM for text generation         |


In [29]:
from transformers import AutoModelForCausalLM

In [30]:
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [31]:
gpt2

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [127]:
sentence = "I like Machine Learning to be able to do things that are not possible in the real world."
# Data Processing technique
token_ids = tokenizer(sentence, return_tensors="pt").input_ids

outputs = gpt2(token_ids).logits[0, -1]

tokenizer.decode(outputs.argmax())

'\n'

In [77]:
outputs = gpt2(token_ids).logits[0,-1]
outputs

tensor([-109.1312, -109.4878, -117.9827,  ..., -119.3269, -123.2292,
        -113.1503], grad_fn=<SelectBackward0>)

In [60]:
token_ids, outputs.argmax()

(tensor([[  40,  588, 4572, 4673,  284]]), tensor(307))

In [131]:
sentence = "I learn machine learning to enhance our understanding of the world around us."

token_ids = tokenizer(sentence, return_tensors="pt").input_ids

outputs = gpt2(token_ids).logits[0, -1]
tokenizer.decode(outputs.argmax())

'\n'

In [154]:
import torch
sentence = "I learn machine learning to enhance human"
token_ids = tokenizer(sentence, return_tensors="pt").input_ids
outputs = gpt2(token_ids).logits[0, -1]
final_logits = torch.topk(outputs, 5) # Feel free to play around with the K

for index in final_logits.indices:
    print(tokenizer.decode(index))

 intelligence
 performance
 cognition
 behavior
 learning


In [155]:
final_logits

torch.return_types.topk(
values=tensor([-117.6033, -117.8474, -118.1260, -118.2189, -118.4656],
       grad_fn=<TopkBackward0>),
indices=tensor([ 4430,  2854, 31119,  4069,  4673]))

In [156]:
torch.softmax(final_logits.values, dim=0).sum()

tensor(1.0000, grad_fn=<SumBackward0>)

In [158]:
def greedy_decode(logits):
    """Return token index with maximum probability."""
    return torch.argmax(logits, dim=-1)

tokenizer.decode(greedy_decode(outputs))

' intelligence'

#### Different Sampling Stratagies

In [252]:
import torch.nn.functional as F

## maximum probability sampling
def greedy_decode(logits):
    """Return token index with maximum probability."""
    return torch.argmax(logits, dim=-1)

# TOP K SAMPLING

def top_k_sampling(logits, k=50):
    """
    keeps only top-k logits, normalize them into probability.
    them sample one token from the filtered distribution.
    """
    values, indices = torch.topk(logits, k)
    probs = F.softmax(values, dim=-1)
    sampled = torch.multinomial(probs, 1)
    return indices[sampled]

# Top-p (Nuecles) Sampling

def top_p_sampling(logits, p=0.9):
    """
    Sort tokens by probability, keep smallest number whose culumative
    probability exceeds threshold p, then sample one token.
    """

    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = sorted_probs.cumsum(dim=-1)

    # Mask token outside nuclues
    mask = cumulative_probs > p
    sorted_logits[mask] = float("-inf")

    # Sample from filtered logits
    filtered_probs = F.softmax(sorted_logits, dim=-1)
    sampled = torch.multinomial(filtered_probs, 1)

    # Return token index in originial vocabulary
    return sorted_indices[sampled]

## Temperature Sampling ##

def temperature_sampling(logits, temperature=1.0):
    """
    Scale logits by temperature before sampling.
    Lower temperature => sharper distribution
    """

    scaled = logits / temperature
    probs = F.softmax(scaled, dim=-1)
    return torch.multinomial(probs, 1)


## Random Sampling ##

def random_sampling(logits):
    """
    Sample dirctly from softmax distribution without filtring
    """

    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1)

# sentence = "Today I decided to go to the local library and find out what was in my wallet."
sentence = "I am really happy becuase I have gone back in"
inputs = tokenizer(sentence, return_tensors="pt")
output = gpt2(**inputs)
logits = output.logits[0, -1]

print(f"Greedy Decode: ", tokenizer.decode([greedy_decode(logits)]))
print(f"Top-K Sampling: ", tokenizer.decode(top_k_sampling(logits, k=10)))
print(f"Top-P-Sampling: ", tokenizer.decode(top_p_sampling(logits, p=0.9)))
print(f"Temp: ", tokenizer.decode(temperature_sampling(logits, temperature=0.001)))
print(f"Radnom: ", tokenizer.decode(random_sampling(logits)))

Greedy Decode:   time
Top-K Sampling:   time
Top-P-Sampling:   time
Temp:   time
Radnom:   time


In [214]:
tokenizer.decode(top_k_sampling(outputs))

' life'

In [215]:
outputs

tensor([-128.7592, -126.0510, -133.6111,  ..., -127.8803, -133.1999,
        -128.2801], grad_fn=<SelectBackward0>)

In [216]:
tokenizer.decode(top_p_sampling(outputs, p=0.9))

' intelligence'

In [217]:
tokenizer.decode(temperature_sampling(outputs, temperature=1.5))

' businessman'

In [219]:
tokenizer.decode(random_sampling(outputs))

' notebooks'

In [253]:
sentence = "I learn machine learning to enhance our understanding of the brain in"
token_ids = tokenizer(sentence, return_tensors="pt").input_ids
outputs = gpt2(token_ids).logits # Raw Unnormlized Score - Values
outputs = torch.softmax(outputs[0, -1], dim=-1)

top10 = torch.topk(outputs, k=10)

for index, value in zip(top10.indices, top10.values):
    print(f"{tokenizer.decode(index)} -- {value:.1%}")

 a -- 14.7%
 the -- 7.7%
 order -- 6.2%
 ways -- 6.2%
 general -- 4.5%
 humans -- 2.9%
 many -- 2.8%
 an -- 2.4%
 our -- 2.3%
 everyday -- 1.7%


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="openai-community/gpt2")

In [ ]:
prompt = "What is machine learning?"
output = pipe(prompt)

In [ ]:
print(output[0]["generated_text"])

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

In [ ]:
model

## Sentiment Analysis

In [ ]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

In [ ]:
type(ds)

In [ ]:
ds

In [ ]:
ds["train"]

In [ ]:
import pandas as pd

In [ ]:
ds["train"].to_pandas()

In [ ]:
my_dataset_df = ds["train"].to_pandas()

In [ ]:
my_dataset_df["text"]

In [ ]:
len(my_dataset_df["text"])

In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")

In [ ]:
classifier("This day is great!")

In [ ]:
classifier("This day is terrible and i am so sad")[0]["label"]

In [ ]:
def score(review_text):
    return classifier(review_text[:500])[0]["label"]

In [ ]:
my_dataset_df["model_prediction"] = my_dataset_df["text"].apply(score)

In [ ]:
my_dataset_df[["label", "model_prediction"]][:20]

In [ ]:
my_dataset_df.iloc[0]

In [ ]:
review = my_dataset_df.iloc[0]["text"]
classifier(review)[0]["label"]

In [ ]:
from transformers import pipeline

In [ ]:
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")

In [ ]:
sentence = "The company reported a strong increase in quarterly revnue, exceeding market expectations."
finbert(sentence)

In [ ]:
sentence = "Shares fell after the firm reported lower-than-expected earnings"
finbert(sentence)

In [ ]:
sentences = ["Strong consumer demand drove record sales across all regions",
             "Supply chain disruptions severly affected production output"]

In [ ]:
finbert(sentences)

### Named Entity Recognition

In [ ]:
sentence = "Apple announced record earnings in the United States on Monday."

In [ ]:
ner = pipeline("ner")

In [ ]:
sentence

In [ ]:
ner(sentence)

In [ ]:
sentence = "I live in UK worked at Facebook after graduating from Harvard"

In [ ]:
ner(sentence)

## Question Answering

In [ ]:
qa_bot = pipeline("question-answering")

In [ ]:
context = """
Financial sentiment analysis is a challenging task due to the specialized
language and lack of labeled data in that domain. General-purpose models are
not effective enough because of the specialized language used in a financial
context. We hypothesize that pre-trained language models can help with this
problem because they require fewer labeled examples and they can be further
trained on domain-specific corpora. We introduce FinBERT, a language model
based on BERT, to tackle NLP tasks in the financial domain. Our results show
improvement in every measured metric on current state-of-the-art results for
two financial sentiment analysis datasets. We find that even with a smaller
training set and fine-tuning only a part of the model, FinBERT outperforms
state-of-the-art machine learning methods.
"""

In [ ]:
question = "What is financial sentiment analysis?"

In [ ]:
qa_bot(question=question, context=context)

In [ ]:
question = "What is FinBERT?"

In [ ]:
result = qa_bot(question=question, context=context)
print(result["answer"])

## Machine Translation

In [ ]:
translater = pipeline("translation_en_to_fr")

In [ ]:
translater("Hello")

In [ ]:
translater("Thanks")

In [ ]:
sentence = "What is your name?"
translater(sentence)[0]["translation_text"]

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("translation", model="facebook/nllb-200-distilled-600M")